# Customer Churn Analysis

## Executive Summary
This project analyzes ecommerce customer behavior to understand what patterns are associated with churn and what signals may help identify at-risk users earlier.

The analysis focuses on customer engagement, session behavior, and purchasing activity. The results show that customers who churn tend to log in less often, spend less time per session, and make fewer purchases. A simple churn risk score was also created to highlight users who may need retention-focused outreach.

## Business Question
Which customer behaviors are most strongly associated with churn, and how can those signals be used to identify at-risk users?

## Dataset
The dataset includes customer demographics and behavioral metrics such as:
- Login frequency
- Session duration
- Pages per session
- Purchase activity
- Engagement metrics
- Churn status

## Project Goal
Identify churn patterns, compare churned vs retained users, and create a simple risk indicator that could support retention decisions.


## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("ecommerce_customer_churn_dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 2. Data Cleaning

This step removes duplicate records, checks missing values, and fills missing numeric values using median imputation.

Median imputation was used because it is less sensitive to extreme values than the mean and provides a simple, practical baseline for exploratory analysis.


In [ ]:
# remove duplicates
df = df.drop_duplicates()

# check missing values
missing_values = df.isnull().sum()
missing_values[missing_values > 0].sort_values(ascending=False)

In [ ]:
# fill numeric missing values with median
df = df.fillna(df.median(numeric_only=True))

# confirm remaining missing values
df.isnull().sum().sum()

## 3. Churn Overview

Start by measuring the overall churn rate and comparing key engagement metrics across churn groups.


In [ ]:
churn_rate = df['Churned'].mean()
print(f"Overall churn rate: {churn_rate:.2%}")

In [ ]:
behavior_summary = df.groupby('Churned')[[
    'Login_Frequency',
    'Session_Duration_Avg',
    'Pages_Per_Session',
    'Total_Purchases',
    'Social_Media_Engagement_Score',
    'Lifetime_Value'
]].mean().round(2)

behavior_summary.index = ['Retained', 'Churned']
behavior_summary

## 4. Visual Analysis

These charts compare average behavior across retained and churned customers.


In [ ]:
df.groupby('Churned')['Login_Frequency'].mean().plot(kind='bar')
plt.title('Average Login Frequency by Churn Status')
plt.xlabel('Churned')
plt.ylabel('Average Login Frequency')
plt.xticks([0, 1], ['No', 'Yes'], rotation=0)
plt.show()

In [ ]:
df.groupby('Churned')['Session_Duration_Avg'].mean().plot(kind='bar')
plt.title('Average Session Duration by Churn Status')
plt.xlabel('Churned')
plt.ylabel('Average Session Duration')
plt.xticks([0, 1], ['No', 'Yes'], rotation=0)
plt.show()

In [ ]:
df.groupby('Churned')['Total_Purchases'].mean().plot(kind='bar')
plt.title('Average Purchases by Churn Status')
plt.xlabel('Churned')
plt.ylabel('Average Total Purchases')
plt.xticks([0, 1], ['No', 'Yes'], rotation=0)
plt.show()

In [ ]:
df.groupby('Churned')['Pages_Per_Session'].mean().plot(kind='bar')
plt.title('Average Pages per Session by Churn Status')
plt.xlabel('Churned')
plt.ylabel('Average Pages per Session')
plt.xticks([0, 1], ['No', 'Yes'], rotation=0)
plt.show()

## 5. Segmentation by Membership Length

This view checks whether customer tenure appears to be related to churn behavior.


In [ ]:
df['membership_bucket'] = pd.qcut(df['Membership_Years'], q=4, duplicates='drop')

membership_churn = df.groupby('membership_bucket')['Churned'].mean().round(3)
membership_churn

In [ ]:
membership_churn.plot(kind='bar')
plt.title('Churn Rate by Membership Length Bucket')
plt.xlabel('Membership Length Bucket')
plt.ylabel('Churn Rate')
plt.xticks(rotation=45)
plt.show()

## 6. Churn Risk Score

A simple churn risk score was created using lower login frequency, lower session duration, and fewer purchases as signals of higher churn risk.

A higher score indicates higher estimated churn risk.


In [ ]:
df['churn_risk_score'] = (
    (df['Login_Frequency'].max() - df['Login_Frequency']) +
    0.5 * (df['Session_Duration_Avg'].max() - df['Session_Duration_Avg']) +
    (df['Total_Purchases'].max() - df['Total_Purchases'])
)

risk_summary = df.groupby('Churned')['churn_risk_score'].mean().round(2)
risk_summary.index = ['Retained', 'Churned']
risk_summary

In [ ]:
df.groupby('Churned')['churn_risk_score'].mean().plot(kind='bar')
plt.title('Average Churn Risk Score by Churn Status')
plt.xlabel('Churned')
plt.ylabel('Average Risk Score')
plt.xticks([0, 1], ['No', 'Yes'], rotation=0)
plt.show()

## 7. Business Interpretation

This analysis suggests that engagement and purchase behavior are useful early signals of churn. In a real product or ecommerce setting, these findings could support:
- retention campaigns for low-engagement users
- reactivation outreach for customers with declining activity
- product experiments focused on increasing session quality and repeat purchases


## 8. Key Insights

- Customers who churn tend to log in less frequently than retained customers.
- Churned users also show lower session duration and fewer purchases on average.
- Pages per session and engagement behavior help reinforce the retention story.
- The churn risk score clearly separates retained and churned customers on average.
- Membership length may also provide useful context for identifying higher-risk groups.

## Limitations

This project uses a static dataset and a simple heuristic churn risk score rather than a predictive machine learning model. In a production setting, the next step would be to build and validate a churn prediction model on unseen data.

## Conclusion

User engagement plays a major role in retention. Metrics such as login frequency, session duration, pages per session, and purchase activity can help identify customers who may be at risk of churning. This type of analysis can help software and ecommerce teams make smarter retention decisions earlier.
